# Multiple dot-product terms per crossbar row: ULTRA, SIMPLER, LOGIC, AUTO, HILO (dense row mapping)

**y = A·x**, matrix row *i* → crossbar row *i* (mod 512). Every row is stored **densely** (all `n` values, zeros included), so a 1176-wide row holds 1176 dot-product terms spread over `ceil(1176 / k)` crossbars.

**Row layout** (every technique, with its own cells): a row holds *k* matrix values, **one shared** x slot (the vector values arrive one at a time from the RAM buffer) and **one** set of working cells (the technique's partial-product / functional / carry cells) reused by every multiplication; only the 64-bit products accumulate at the end of the row. Example, HILO (209 cells = inputs 64 + working 81 + output 64):

| stage | matrix | x slot | working cells | products | unused |
|---|---|---|---|---|---|
| 1: m1·a → 1a | 4×32 | a | 81 | 1a | 207 |
| 2: m2·b → 2b | 4×32 | b | 81 | 2b 1a | 143 |
| 3: m3·c → 3c | 4×32 | c | 81 | 3c 2b 1a | 79 |
| 4: m4·d → 4d | 4×32 | d | 81 | 4d 3c 2b 1a | 15 |
| final | 4×32 | sum of the 4 products stored in the freed 128 cells | | 4d 3c 2b 1a | 0 |

k = ⌊(band·512 − 32 − working cells) / 96⌋ for each technique (HILO 4, AUTO 4, LOGIC 3; SIMPLER and ULTRA need a band of 2 and 30 rows and pay the row-boundary carry copies). Partial sums of a matrix row are added across its crossbars with a **tree** (⌈log₂ C⌉ levels of copy + add), each technique with its own adder.
Run the cells top to bottom.

## 1. Parameters

In [ ]:
XBAR_ROWS = 512
ROW_CELLS = 512
N = 32                     # precision
SET_MODE = "prep"          # HiLo multiplication SETs: "prep" (PREP parallel SET) or "greedy"
MUL_SOURCE = "paper"       # HILO / AUTO multiplication: "paper" = adder gates from the HiLo paper table (HILO exact
                           # search 8,081, AUTO greedy 8,462) + carry-copy NOTs + SETs computed here (pp / functional /
                           # carry cell reuse); "computed" = all three from the adder list + lookup table
BORROW_SPARE = True        # HILO / AUTO: the cells a row leaves unused after its k products (HILO 15, AUTO 4)
                           # are used as extra functional cells during the multiplications (fewer SETs)
EXTRA_POOL = 0             # extra functional cells a multiplication may borrow
LINKS = "hier"             # crossbar-to-crossbar copies: "hier" = hierarchical bus (H-tree of buses),
                           # "shared" = one bus for every crossbar, "parallel" = a link per crossbar pair
BUS_FANOUT = 16            # "hier": crossbars per level-1 bus (and level-1 blocks per level-2 bus, ...)
X_SOURCE = "buffer"        # vector values come one at a time from the RAM buffer over the bus ("buffer"),
                           # or each crossbar has its own link ("parallel")
X_ORDER = "sequential"     # vector values arrive one after another: the crossbars of one matrix row multiply
                           # one after another, only matrix rows run in parallel ("parallel": all crossbars at once)
X_WRITE_CYCLES = 2         # write the broadcast x value into the shared x slot of all rows (1-bits, then 0-bits)
T_CYCLE_NS = 1.0           # ns per cycle
DROP_ZERO_COLUMNS = True   # all-zero matrix columns are discarded (their x is never needed)
LUT_FILE = "lookuptable_with_functional_cells.csv"   # needs columns col1..col8, weight, Cost, functional_cells

## 2. Lookup table (functional cells + adder costs)
Uses `lookuptable_with_functional_cells.csv` if it is already in the Colab files; otherwise asks you to upload it.

In [ ]:
import os
if not os.path.exists(LUT_FILE):
    from google.colab import files
    print(f"Upload {LUT_FILE}")
    up = files.upload()
    name = next(iter(up))
    if name != LUT_FILE:
        os.replace(name, LUT_FILE)
print(LUT_FILE, "found:", os.path.exists(LUT_FILE))

## 3. Modules
`hilo_cost.py` (HiLo adder list, cell segments, greedy SETs), `hilo_prep.py` (PREP parallel SET), `own_adder.py` (HiLo's own adder cover from the lookup table), `dense_rowmap.py` (row layout, dense row mapping, tree reduction, latency / energy / area for any technique).

In [ ]:
%%writefile hilo_cost.py
"""
HiLo / AUTO cost of ONE 32x32 multiplication, from the adder list and the
functional-cells lookup table (same rules as hilo_final_latency_and_energy.ipynb).

Region of one multiplication:
  | ~A, ~B (inputs) | partial product (reused) | functional (reused) | carry | output |
  fixed cells = inputs 2N + pp segment + carry peak + output 2N
  functional  = max functional_cells (lookup table) over the adders used
  cells per multiplication = fixed + functional        (HILO -> 209, AUTO -> 220)

Per adder step:
  1. SET the partial-product segment                         1 cycle
  2. copy the carries it needs into the pp cells             2 NOTs per carry bit
  3. generate its new partial products                       pp_ops per bit (0 or 3)
  4. run its netlist in the functional cells                 'cost' gates
     + SET batches: ceil(cost / pool), pool = functional cells it may use
  5. lowest output bit -> Output (final); others -> Carry cells
One gate per cycle per row; all rows of a crossbar run the same gate in parallel.
"""
import math, re
from collections import defaultdict
import pandas as pd

N = 32
COPY_GATES = 2          # NOT + NOT per copied bit

HILO_TEXT = """
adder (2,3) W=4 cost=20 at column 1
adder (5,1,3) W=5 cost=42 at column 3
adder (6,4,4) W=5 cost=81 at column 4
adder (5,1,3,3) W=6 cost=60 at column 6
adder (10,8,8) W=6 cost=173 at column 7
adder (13) W=4 cost=79 at column 10
adder (15,3) W=5 cost=114 at column 11
adder (13) W=4 cost=79 at column 12
adder (17,7,3) W=6 cost=179 at column 13
adder (11,2) W=4 cost=77 at column 14
adder (15,3) W=5 cost=114 at column 15
adder (17,7) W=5 cost=158 at column 16
adder (15) W=4 cost=93 at column 17
adder (23,7,3) W=6 cost=230 at column 18
adder (17,7) W=5 cost=158 at column 19
adder (15) W=4 cost=93 at column 20
adder (25,4,7) W=6 cost=250 at column 21
adder (23,7,3) W=6 cost=230 at column 22
adder (15) W=4 cost=93 at column 23
adder (25,7) W=6 cost=219 at column 24
adder (23,7,4,1) W=6 cost=242 at column 25
adder (25,3) W=5 cost=191 at column 26
adder (25,7,3) W=6 cost=243 at column 27
adder (25,7,4) W=6 cost=249 at column 28
adder (25,7,4) W=6 cost=249 at column 29
adder (25,7,4,1) W=6 cost=255 at column 30
adder (25,7,4,1) W=6 cost=255 at column 31
adder (25,13,1,1) W=6 cost=281 at column 32
adder (17,7) W=5 cost=158 at column 33
adder (25,7,4,1) W=6 cost=255 at column 34
adder (25,3) W=5 cost=191 at column 35
adder (25,13,3) W=6 cost=288 at column 36
adder (17,9,7) W=6 cost=223 at column 37
adder (17,7,4,2) W=6 cost=200 at column 38
adder (15,8) W=5 cost=150 at column 39
adder (15) W=4 cost=93 at column 40
adder (25,1,1) W=5 cost=184 at column 41
adder (25,7,4,1) W=6 cost=255 at column 42
adder (17,7) W=5 cost=158 at column 43
adder (11,6,6,2) W=6 cost=159 at column 44
adder (15,8,8) W=6 cost=207 at column 45
adder (7,4) W=4 cost=61 at column 46
adder (7,4,4,4) W=6 cost=112 at column 47
adder (15,8,8) W=6 cost=207 at column 48
adder (7,4) W=4 cost=61 at column 49
adder (15,8,8) W=6 cost=207 at column 51
adder (7,4) W=4 cost=61 at column 52
adder (11,6,6,2) W=6 cost=159 at column 54
adder (5,1) W=3 cost=28 at column 55
adder (3,6,6,3) W=6 cost=104 at column 56
adder (3,4,3,1) W=5 cost=54 at column 59
"""

AUTO_HEIGHTS = [2, 4, 5, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 21, 22, 23, 24, 25, 26, 27,
                28, 29, 30, 31, 32, 33, 34, 35, 36, 36, 35, 34, 33, 32, 31, 30, 29, 28, 27, 25, 24,
                23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12, 10, 9, 8, 7, 6, 5, 3, 2]
AUTO_WEIGHTS = [2, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6,
                6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4,
                4, 4, 3, 3, 3, 2, 2]
AUTO_COSTS = [5, 16, 22, 36, 45, 49, 58, 74, 79, 88, 93, 104, 109, 121, 126, 139, 148, 153, 166,
              173, 187, 192, 199, 203, 213, 220, 229, 233, 249, 254, 265, 265, 254, 249, 233, 229,
              220, 213, 203, 199, 192, 173, 166, 153, 148, 139, 134, 126, 121, 109, 104, 93, 88,
              79, 74, 58, 49, 45, 36, 31, 22, 11, 5]


def parse_adders(txt):
    return [(list(map(int, m.group(1).split(","))), int(m.group(2)), int(m.group(3)), int(m.group(4)))
            for m in re.finditer(r"adder \(([\d,]+)\) W=(\d+) cost=(\d+) at column (\d+)", txt)]


HILO = parse_adders(HILO_TEXT)
AUTO = [([h], w, c, i + 1) for i, (h, w, c) in enumerate(zip(AUTO_HEIGHTS, AUTO_WEIGHTS, AUTO_COSTS))]
TECHS = {"HILO": HILO, "AUTO": AUTO}

# partial-product bits per column (bit 0 = a0b0 needs no adder): 2, 3, ..., 32, 31, ..., 1
PP_INIT = {c: (c + 1 if c <= N - 1 else 2 * N - 1 - c) for c in range(1, 2 * N - 1)}


def load_functional_cells(path):
    """(column vector, weight) -> functional cells, from lookuptable_with_functional_cells.csv"""
    lut = pd.read_csv(path)
    fc = {}
    for _, r in lut.iterrows():
        v = [int(r[f"col{i}"]) for i in range(1, 9)]
        while v and v[-1] == 0:
            v.pop()
        if pd.notna(r["functional_cells"]):
            fc[(tuple(v), int(r["weight"]))] = int(r["functional_cells"])
    return fc


def schedule(G):
    """Column sweep: fresh partial products vs. copied carries per adder; peak waiting carries."""
    pp_left = defaultdict(int, PP_INIT)
    waiting = defaultdict(int)
    steps, peak = [], 0
    for k, w, cost, col in G:
        copied = new_pp = 0
        for j, kj in enumerate(k):
            take = min(waiting[col + j], kj)
            waiting[col + j] -= take
            copied += take
            new_pp += kj - take
            pp_left[col + j] -= kj - take
            if pp_left[col + j] < 0:
                raise ValueError(f"adder at column {col} takes more bits than exist")
        for j in range(w):
            waiting[col + j] += 1
        for c in list(waiting):
            if c <= col and pp_left[c] == 0 and waiting[c] == 1:
                waiting[c] = 0
        peak = max(peak, sum(waiting.values()))
        steps.append(dict(inputs=sum(k), copied=copied, new_pp=new_pp, W=w, cost=cost))
    if any(v for v in pp_left.values()):
        raise ValueError("some partial products are never added")
    return steps, peak


def multiply_cost(G, fc, pp_ops=0, extra_pool=0):
    """Cost of one multiplication in its own region (fixed + functional cells).

    pool = functional cells the adders may use = this multiplication's own functional
    cells + extra_pool (e.g. spare cells of the row it may borrow). SET batches per adder
    = ceil(cost / pool).
    """
    steps, carry_peak = schedule(G)
    seg = dict(inputs=2 * N, partial_product=max(s["inputs"] for s in steps),
               carry=carry_peak, output=2 * N)
    fixed = sum(seg.values())
    missing = [k for k, w, _, _ in G if (tuple(k), w) not in fc]
    if missing:
        raise KeyError(f"adders not in the functional-cells table: {missing[:5]}")
    need = max(fc[(tuple(k), w)] for k, w, _, _ in G)
    pp_cells = 2 if pp_ops == 3 else 0          # 2 cells hold ~a, ~b while a pp is made
    pool = need + extra_pool
    ops = dict(gates=0, copy_NOT=0, carries_copied=0, pp=0, set_pp=0, set_func=0)
    for s in steps:
        ops["gates"] += s["cost"]
        ops["copy_NOT"] += COPY_GATES * s["copied"]
        ops["carries_copied"] += s["copied"]
        ops["pp"] += pp_ops * s["new_pp"]
        ops["set_pp"] += 1
        ops["set_func"] += max(1, math.ceil(s["cost"] / pool))
    ops["pp"] += pp_ops                          # a0*b0
    ops["set_total"] = ops["set_pp"] + ops["set_func"] + 1      # +1 initial SET of the region
    evals = ops["gates"] + ops["copy_NOT"] + ops["pp"]
    return dict(seg=seg, fixed=fixed, functional=need, cells=fixed + need + pp_cells,
                pool=pool, steps=len(steps), ops=ops, evals=evals,
                cycles=evals + ops["set_total"])


In [ ]:
%%writefile hilo_prep.py
"""
PREP parallel cell preparation (Rashed et al., "Execution Sequence Optimization for
Processing In-Memory using Parallel Data Preparation", DAC 2024) applied to the
HiLo / AUTO 32-bit multiplication of hilo_cost.py.

Same multiplication as hilo_cost.multiply_cost (same adder list, carry reuse, copies,
cells), only the SET rule changes:
  greedy (hilo_cost) : every adder step SETs the partial-product segment (1) and its
                       functional cells (ceil(cost/pool), at least 1)
  PREP   (this file) : SETs are postponed until no prepared cell is left; then ALL
                       expired cells of the region (partial-product + functional) are SET
                       together in ONE parallel SET cycle. Leftover prepared cells carry
                       over to the next adder step.
Carry cells are not SET separately (same as hilo_cost).
"""
import math
from hilo_cost import N, COPY_GATES, schedule


def multiply_cost_prep(G, fc, pp_ops=0, extra_pool=0):
    steps, carry_peak = schedule(G)
    seg = dict(inputs=2 * N, partial_product=max(s["inputs"] for s in steps),
               carry=carry_peak, output=2 * N)
    fixed = sum(seg.values())
    missing = [k for k, w, _, _ in G if (tuple(k), w) not in fc]
    if missing:
        raise KeyError(f"adders not in the functional-cells table: {missing[:5]}")
    func_cells = max(fc[(tuple(k), w)] for k, w, _, _ in G)
    pp_cells = 2 if pp_ops == 3 else 0          # 2 cells hold ~a, ~b while a pp is made
    pool = func_cells + extra_pool
    P = seg["partial_product"]

    ops = dict(gates=0, copy_NOT=0, carries_copied=0, pp=0, set_parallel=0)
    pp_ready, f_ready = P, pool                  # prepared cells left after the initial SET
    for s in steps:
        ops["gates"] += s["cost"]
        ops["copy_NOT"] += COPY_GATES * s["copied"]
        ops["carries_copied"] += s["copied"]
        ops["pp"] += pp_ops * s["new_pp"]
        need = s["inputs"]                       # pp cells this adder writes (copied carries + new pp)
        if pp_ready < need:                      # no prepared pp cells: one parallel SET
            ops["set_parallel"] += 1
            pp_ready, f_ready = P, pool
        pp_ready -= need
        if s["cost"] <= f_ready:                 # gates fit in the prepared functional cells
            f_ready -= s["cost"]
        else:                                    # run out: parallel SETs of all expired cells
            rem = s["cost"] - f_ready
            k = math.ceil(rem / pool)
            ops["set_parallel"] += k
            f_ready = k * pool - rem
            pp_ready = P - need                  # expired pp cells of earlier steps prepared too
    ops["pp"] += pp_ops                          # a0*b0
    ops["set_total"] = ops["set_parallel"] + 1   # +1 initial SET of the region
    evals = ops["gates"] + ops["copy_NOT"] + ops["pp"]
    return dict(seg=seg, fixed=fixed, functional=func_cells, cells=fixed + func_cells + pp_cells,
                pool=pool, steps=len(steps), ops=ops, evals=evals, set_mode="prep",
                cycles=evals + ops["set_total"])


In [ ]:
%%writefile own_adder.py
"""
Each technique's own adder for the reduction, copied from reduction_eval.py in
hilo_final_latency_and_energy.ipynb (load_lib and add_cover unchanged).
Adding k numbers of WIDTH bits = a bit block with k bits in every column; the column-sweep
A* search finds its cheapest cover:  HILO = full adder library, AUTO = single-column adders.
cycles = gates + 1 SET per adder.
"""
import csv, heapq, math
from fractions import Fraction

WIDTH = 32                  # accumulator width


def load_lib(path, single_column=False):
    lib = []
    with open(path, newline="") as f:
        for r in csv.DictReader(f):
            v = [int(r[f"col{i}"]) for i in range(1, 9)]
            while v and v[-1] == 0:
                v.pop()
            if single_column and len(v) != 1:
                continue
            lib.append((tuple(v), int(r["weight"]), int(round(float(r["Cost"])))))
    lib.sort(key=lambda a: a[0][0])
    return lib


def add_cover(lib, k, width=WIDTH, goal="relaxed"):
    """Cheapest cover of k numbers of `width` bits (column-sweep A*, as in hilo_sweep.py)."""
    C = width + 1 + (math.ceil(math.log2(k)) if k > 1 else 0)
    init = [k if c < width else 0 for c in range(C + 8)]
    suffix = [0] * (C + 9)
    for c in range(C + 7, -1, -1):
        suffix[c] = suffix[c + 1] + init[c]
    rho = min(Fraction(c, sum(v) - w) for v, w, c in lib if sum(v) - w > 0)
    col = lambda k0, win, c: win[c - k0] if c - k0 < 6 else init[c]

    def h(k0, win):
        if k0 >= C:
            return 0
        ex = sum(win) + suffix[k0 + 6] - (C - k0)
        return 0 if ex <= 0 else -((-ex * rho.numerator) // rho.denominator)

    k0 = next(c for c in range(C) if init[c] >= 2)
    start = (k0, tuple(init[k0:k0 + 6]))
    g, parent, heap = {start: 0}, {start: None}, [(h(*start), 0, start)]
    while heap:
        _, gs, st = heapq.heappop(heap)
        if gs > g[st]:
            continue
        kk, win = st
        if kk >= C:
            steps = 0
            while parent[st]:
                st = parent[st][0]; steps += 1
            return gs, steps
        for v, w, cost in lib:
            if v[0] > win[0]:
                break
            if kk + w - 1 >= C or any(v[j] > win[j] for j in range(1, len(v))):
                continue
            t = [win[j] - (v[j] if j < len(v) else 0) + (1 if j < w else 0) for j in range(6)]
            nk, dead = kk, False
            while nk < C and col(kk, t, nk) < 2:
                if goal == "paper" and col(kk, t, nk) == 0:
                    dead = True
                    break
                nk += 1
            if dead:
                continue
            nwin = tuple(col(kk, t, c) if c < C else 0 for c in range(nk, nk + 6))
            ns, ng = (nk, nwin), gs + cost
            if ng < g.get(ns, 1 << 60):
                g[ns], parent[ns] = ng, (st, None)
                heapq.heappush(heap, (ng + h(nk, nwin), ng, ns))
    return None, 0


def own_adders(lookup, k=2, width=WIDTH):
    """{'HILO': {...}, 'AUTO': {...}} cost of adding k numbers with each technique's adders."""
    out = {}
    for t, single in (("HILO", False), ("AUTO", True)):
        gates, steps = add_cover(load_lib(lookup, single_column=single), k, width)
        out[t] = dict(gates=gates, adders=steps, cycles=gates + steps)
    return out


In [ ]:
%%writefile dense_rowmap.py
"""
Dense ROW mapping of y = A x with multiple dot-product terms per crossbar row (HiLo).

Row layout (one crossbar row, ROW cells), for k terms of one matrix row:
  | m_1 | m_2 | ... | m_k | x slot | shared working cells | free | p_k | ... | p_2 | p_1 |
     N     N           N      N      (pp + functional + carry)        2N          2N    2N
  stage s (s = 1..k): the vector values arrive one at a time from the RAM buffer; x_s is written into the ONE
  shared x slot (same value for every row of the crossbar, because all rows of crossbar c hold the same matrix
  columns), m_s * x_s runs in the shared working cells, the 2N-bit
  product p_s is stored at the end of the row.  Finally the k products are added (tree order) and the sum is
  stored in the cells freed by the x slot, the working cells and the free cells.
  k = floor((ROW - N - shared) / (N + 2N))          (HiLo: (512 - 32 - 81) / 96 = 4)

Every technique uses the same layout with its own cells (shared working cells = its partial-product /
functional / carry cells); a technique whose multiplication is wider than one row uses a band of
ceil(cells / ROW) rows (SIMPLER 2, ULTRA 30): k = floor((band*ROW - N - shared) / 3N).

Dense: every matrix row stores ALL n_cols values (zeros included), so matrix row i uses crossbar row
(i mod XB_ROWS) of ceil(n_cols / k) crossbars. All-zero matrix columns can be dropped (their x is never needed).
Partial sums of a row are added across its crossbars with a tree: ceil(log2 C) levels, each = copy + add.
The vector values come from the RAM buffer one after another, so the crossbars of one matrix row multiply one
after another; only different matrix rows (crossbar rows / row groups) compute in parallel.
"""
import math

N = 32
BUS_BITS = 256
P = dict(xbar=1.20, driver=0.65, sense_amp=0.58, local_bus=4.66, global_bus=26.0)   # mW (AUTO Table III)
ACTIVE = {
    "x write + multiply": (["xbar", "driver"], False),
    "in-row add": (["xbar", "driver"], False),
    "merge (copies)": (["sense_amp", "local_bus"], True),
    "tree add": (["xbar", "driver"], False),
    "readout": (["sense_amp", "local_bus"], True),
}


def bus(bits):
    return (2 + math.ceil(bits / BUS_BITS)) if bits else 0     # read + transfer + write


def shared_terms_per_row(row_cells, shared_cells, n=N, rows_per_band=1):
    """Terms a band of rows_per_band crossbar rows holds: k*N + N + shared + k*2N <= rows_per_band*ROW."""
    return (rows_per_band * row_cells - n - shared_cells) // (n + 2 * n)


def stages(k, row_cells, shared_cells, n=N):
    """Cells used / unused at every stage of one crossbar row (shared-working-cell layout)."""
    out = []
    fixed = k * n + n + shared_cells
    for s in range(1, k + 1):
        used = fixed + s * 2 * n
        out.append(dict(stage=f"stage {s}", matrix=k * n, x_slot=n, shared=shared_cells,
                        products=s * 2 * n, unused=row_cells - used, used=used))
    free = row_cells - fixed - k * 2 * n
    out.append(dict(stage="final (sum)", matrix=k * n, x_slot=0, shared=0, products=k * 2 * n,
                    unused=0, used=row_cells, sum_region=n + shared_cells + free))
    return out


def hier_merge(C, row_groups, acc_bits, fanout=16):
    """Tree copies over a hierarchical bus (H-tree of buses): crossbars are placed in order (row group by row
    group); every bus_fanout crossbars share a level-1 bus, every bus_fanout level-1 blocks share a level-2 bus,
    and so on. A copy a -> b uses the lowest bus that covers both; copies on different buses run in parallel,
    copies on the same bus go one after another. Cost of a tree level = the busiest bus of that level."""
    levels, merge = 0, 0
    alive = [list(range(g * C, (g + 1) * C)) for g in range(len(row_groups))]
    while len(alive[0]) > 1:
        load = {}
        for g, xs in enumerate(alive):
            bits = row_groups[g] * acc_bits               # partial sums of every matrix row the crossbar holds
            for i in range(0, len(xs) - 1, 2):
                a, b = xs[i + 1], xs[i]                   # a sends to b
                h, span = 1, fanout
                while a // span != b // span:
                    h, span = h + 1, span * fanout
                key = (h, a // span)
                load[key] = load.get(key, 0) + bits
        merge += max(bus(v) for v in load.values())
        alive = [xs[0::2] for xs in alive]
        levels += 1
    return levels, merge


def evaluate_dense(n_rows, n_cols, k, t_mul, t_add_row, t_add_acc, acc_bits, cells_for, xbar_rows=512,
                   row_cells=512, links="parallel", x_write_cycles=2, t_cycle_ns=1.0, x_source="buffer",
                   rows_per_band=1, bus_fanout=16, x_order="sequential"):
    """Latency / energy / area of y = A x, dense row mapping, k terms per band of rows_per_band crossbar rows
    (1 for HILO / AUTO / LOGIC; a technique whose multiplication is wider than a row folds it over several rows).
    cells_for(j) = cells one band uses when it holds j terms (for the area utilization)."""
    bands = xbar_rows // rows_per_band                 # matrix rows one crossbar holds
    C = math.ceil(n_cols / k)                          # crossbars per row group
    k_last = n_cols - (C - 1) * k                      # terms in the last crossbar of a row
    row_groups = [min(bands, n_rows - g) for g in range(0, n_rows, bands)]
    n_x = C * len(row_groups)
    # 1. multiplications. x_order = "sequential": the vector values arrive from the RAM buffer one after another
    #    (x_1, x_2, ..., x_n); x_j goes to the crossbar that holds column j (crossbar ceil(j/k) of every row group),
    #    is written into its shared x slot (x_write_cycles) and multiplied by all its rows in parallel. So the
    #    crossbars of one matrix row compute one after another (crossbar 1 does stages 1..k, then crossbar 2, ...);
    #    only the matrix rows (crossbar rows, row groups) run in parallel.  A crossbar adds its own k products while
    #    the next crossbar multiplies, so only the last crossbar's in-row add is on the critical path.
    #    x_order = "parallel": every crossbar gets its values at the same time (k stages in total).
    in_row_adds = (C - 1) * (k - 1) + max(k_last - 1, 0)          # in-row additions per row group
    if x_order == "sequential":
        x_in = bus(N)                                             # one value per step, broadcast to the row groups
        t = {"x write + multiply": n_cols * (x_in + x_write_cycles + t_mul)}
        t["in-row add"] = max(k_last - 1, 0) * t_add_row          # only the last crossbar is exposed
        work = {"x write + multiply": len(row_groups) * t["x write + multiply"],     # crossbar-cycles of work
                "in-row add": len(row_groups) * in_row_adds * t_add_row}
    else:
        x_in = bus(C * N) if x_source == "buffer" else bus(N)
        t = {"x write + multiply": k * (x_in + x_write_cycles + t_mul)}
        t["in-row add"] = (k - 1) * t_add_row
        work = {"x write + multiply": n_x * t["x write + multiply"], "in-row add": n_x * t["in-row add"]}
    # 3. tree across the C crossbars of a row group: ceil(log2 C) levels of copy + add
    if links == "hier":
        levels, merge = hier_merge(C, row_groups, acc_bits, bus_fanout)
    else:
        levels, merge, alive = 0, 0, C
        while alive > 1:
            senders = alive // 2
            if links == "parallel":
                merge += bus(bands * acc_bits)            # all pairs copy at the same time
            else:                                         # one shared bus for every crossbar
                merge += bus(senders * sum(row_groups) * acc_bits)
            alive = alive - senders
            levels += 1
    t["merge (copies)"] = merge
    t["tree add"] = levels * t_add_acc
    t["readout"] = xbar_rows + bus(n_rows * acc_bits)
    # energy counts only the crossbars that are active: a row group's C-1 copies (sender reads, bus, receiver
    # writes), its C-1 tree additions (one per receiving crossbar) and the readout of the crossbar holding y
    work["merge (copies)"] = sum((C - 1) * bus(rg * acc_bits) for rg in row_groups)
    work["tree add"] = len(row_groups) * (C - 1) * t_add_acc
    work["readout"] = len(row_groups) * t["readout"]
    e = {}
    for ph, cyc in t.items():
        comps, uses_bus = ACTIVE[ph]
        xbar_cycles = work.get(ph, n_x * cyc)          # crossbar-cycles of activity in this phase
        e[ph] = (xbar_cycles * sum(P[c] for c in comps) + (P["global_bus"] * cyc if uses_bus else 0.0)) * t_cycle_ns  # pJ
    total = sum(t.values())
    area = n_x * xbar_rows * row_cells
    used = n_rows * ((C - 1) * cells_for(k) + cells_for(k_last))       # cells holding data
    return dict(k=k, rows_per_band=rows_per_band, crossbars_per_group=C, row_groups=len(row_groups),
                crossbars=n_x, tree_levels=levels,
                cycles=t, total_cycles=total, latency_us=total * t_cycle_ns / 1e3,
                energy_pJ=e, energy_uJ=sum(e.values()) / 1e6, area_cells=area,
                area_utilization=used / area)


## 4. Every technique: cells, row layout, multiplication and adders

In [ ]:
import importlib, math, pandas as pd
import hilo_cost, hilo_prep, own_adder, dense_rowmap as dr
for mod in (hilo_cost, hilo_prep, own_adder, dr):
    importlib.reload(mod)
TECHS5 = ["ULTRA", "SIMPLER", "LOGIC", "AUTO", "HILO"]
ROW_COPY_CYCLES = 2 * 2 * N        # folded multiplication: carries copied to the next row, 2 NOTs x 2N bits

# ---- one 32-bit multiplication of every technique: cells, shared working cells, cycles ----
FC = hilo_cost.load_functional_cells(LUT_FILE)
TECH = {}
def _mul(G, extra):
    return (hilo_prep.multiply_cost_prep(G, FC, extra_pool=extra) if SET_MODE == "prep"
            else hilo_cost.multiply_cost(G, FC, extra_pool=extra))

for t in ("HILO", "AUTO"):                         # adder list + functional-cells table
    G = hilo_cost.TECHS[t]
    m = _mul(G, EXTRA_POOL)
    shared = m["seg"]["partial_product"] + m["functional"] + m["seg"]["carry"]
    # BORROW_SPARE: the cells a row leaves unused after its k products (HILO 15, AUTO 4) are free during all
    # k multiplications (the final addition only starts after the last one), so they serve as extra
    # functional cells: fewer parallel SETs, same cells and same terms per row
    k0 = dr.shared_terms_per_row(ROW_CELLS, shared, N, 1)
    spare = ROW_CELLS - (k0 * 3 * N + N + shared) if BORROW_SPARE else 0
    if spare:
        m = _mul(G, EXTRA_POOL + spare)
    TECH[t] = dict(cells=m["cells"], shared=shared, borrowed=spare,
                   t_mul=m["cycles"], gates=m["ops"]["gates"], copies=m["ops"]["copy_NOT"], sets=m["ops"]["set_total"],
                   source=f"adder list + lookup table ({SET_MODE} SET)")
# LOGIC [TODAES'25]: inputs 64 + PP row 32 + IS 33 + output 64 + functional 42 = 235 cells;
# 10,046 functional cycles + cell preparation (PREP paper, Table 1: 1,271 with PREP, 5,549 greedy)
TECH["LOGIC"] = dict(cells=235, shared=N + (N + 1) + 42, t_mul=10046 + (1271 if SET_MODE == "prep" else 5549),
                     source="LOGIC / PREP paper Table 1")
# SIMPLER [ISCAS'18] and ULTRA [DAC'17]: published totals; working cells = total - inputs - output
TECH["SIMPLER"] = dict(cells=20 * N - 5, shared=20 * N - 5 - 4 * N, t_mul=12870, source="published")
TECH["ULTRA"] = dict(cells=15071, shared=15071 - 4 * N, t_mul=15007, source="published")
# HiLo paper table: adder-gate cycles of one 32-bit multiplication (HILO exact search, AUTO greedy). They do not
# include the carry-copy NOTs or the SETs for reusing the pp / functional / carry cells; those are added from the
# computation above (multiplication = adder gates + carry-copy NOTs + SETs)
PAPER_GATES = {"HILO": 8081, "AUTO": 8462}
if MUL_SOURCE == "paper":
    for t, v in PAPER_GATES.items():
        TECH[t]["gates"] = v
        TECH[t]["t_mul"] = v + TECH[t]["copies"] + TECH[t]["sets"]
        TECH[t]["source"] = (f"HiLo paper table ({'exact search' if t == 'HILO' else 'greedy'}) gates "
                             f"+ carry copies + {SET_MODE} SETs")
for t, d in TECH.items():
    d["rows_per_band"] = math.ceil(d["cells"] / ROW_CELLS)       # rows one multiplication needs
    d["k"] = dr.shared_terms_per_row(ROW_CELLS, d["shared"], N, d["rows_per_band"])
    d["fold"] = (d["rows_per_band"] - 1) * ROW_COPY_CYCLES        # row-boundary carry copies
    d["t_mul_eff"] = d["t_mul"] + d["fold"]
    d["unused_last_stage"] = d["rows_per_band"] * ROW_CELLS - (d["k"] * 3 * N + N + d["shared"])

# ---- each technique's own 2-input adder at any width ----
LIBS = {"HILO": own_adder.load_lib(LUT_FILE), "AUTO": own_adder.load_lib(LUT_FILE, single_column=True)}
_ADD = {}
def add_cycles(t, width):
    key = (t, width)
    if key not in _ADD:
        if t in LIBS:                              # cheapest cover from the lookup table (gates + 1 SET per adder)
            gates, adders = own_adder.add_cover(LIBS[t], 2, width)
            _ADD[key] = gates + adders
        elif t == "LOGIC":                         # 322 functional + 41 (PREP) / 179 (greedy) preparation per 32 bits
            _ADD[key] = round((322 + (41 if SET_MODE == "prep" else 179)) * width / 32) + math.ceil(width / 8)
        else:                                      # ULTRA, SIMPLER: MAGIC 12N+1
            _ADD[key] = 12 * width + 1
    return _ADD[key]

display(pd.DataFrame([{"tech": t, "cells per mult": TECH[t]["cells"], "shared working cells": TECH[t]["shared"],
                       "rows per band": TECH[t]["rows_per_band"], "terms per band (k)": TECH[t]["k"],
                       "unused after last stage": TECH[t]["unused_last_stage"],
                       "spare cells borrowed as functional": TECH[t].get("borrowed", 0),
                       "adder gates": TECH[t].get("gates", "-"), "carry-copy NOTs": TECH[t].get("copies", "-"),
                       "SETs": TECH[t].get("sets", "-"),
                       "cycles per mult": TECH[t]["t_mul"], "row-boundary copies": TECH[t]["fold"],
                       f"{2 * N}-bit add (cycles)": add_cycles(t, 2 * N), "source": TECH[t]["source"]}
                      for t in TECHS5]).style.hide(axis="index"))

# ---- HiLo row, stage by stage (matches the drawing) ----
print("\nHILO: one crossbar row, stage by stage (cells)")
display(pd.DataFrame(dr.stages(TECH["HILO"]["k"], ROW_CELLS, TECH["HILO"]["shared"], N)).style.hide(axis="index"))


## 5. eris1176

In [ ]:
import os, subprocess, math, numpy as np, pandas as pd, scipy.io, scipy.sparse as sp
os.makedirs("matrices", exist_ok=True)
ERIS = "matrices/eris1176.mtx.gz"
if not os.path.exists(ERIS):
    subprocess.run(["curl", "-sSL", "-o", ERIS,
                    "https://math.nist.gov/pub/MatrixMarket2/Harwell-Boeing/smtape/eris1176.mtx.gz"])
A = sp.csc_matrix(scipy.io.mmread(ERIS))
n_rows, n_cols = A.shape
zero_cols = int((np.diff(A.indptr) == 0).sum())
cols_kept = n_cols - zero_cols if DROP_ZERO_COLUMNS else n_cols
print(f"eris1176: {n_rows} x {n_cols}, {A.nnz} nonzeros, {zero_cols} all-zero columns -> {cols_kept} terms per row")


def run(t, n_rows, cols_kept):
    d = TECH[t]
    acc = 2 * N + math.ceil(math.log2(cols_kept))                    # width of the partial sums
    cells_for = lambda j: j * N + N + d["shared"] + j * 2 * N
    return dr.evaluate_dense(n_rows, cols_kept, d["k"], d["t_mul_eff"], add_cycles(t, 2 * N), add_cycles(t, acc),
                             acc, cells_for, xbar_rows=XBAR_ROWS, row_cells=ROW_CELLS, links=LINKS,
                             x_write_cycles=X_WRITE_CYCLES, t_cycle_ns=T_CYCLE_NS, x_source=X_SOURCE,
                             rows_per_band=d["rows_per_band"], bus_fanout=BUS_FANOUT,
                             x_order=X_ORDER)


ERES = {t: run(t, n_rows, cols_kept) for t in TECHS5}
rows = []
for t in TECHS5:
    r = ERES[t]
    rows.append({"tech": t, "terms per band": r["k"], "crossbars": r["crossbars"], "tree levels": r["tree_levels"],
                 **{f"{ph} (cyc)": v for ph, v in r["cycles"].items()},
                 "latency (us)": r["latency_us"], "energy (uJ)": r["energy_uJ"],
                 "area (M cells)": r["area_cells"] / 1e6, "area utilization (%)": 100 * r["area_utilization"]})
print("\ny = A x on eris1176, dense row mapping, shared x slot and working cells")
display(pd.DataFrame(rows).style.format({"latency (us)": "{:.3f}", "energy (uJ)": "{:.3f}",
                                         "area (M cells)": "{:,.2f}", "area utilization (%)": "{:.2f}"}).hide(axis="index"))
print("\nHILO improvement on eris1176 (positive % = HILO is lower)")
display(pd.DataFrame([{"vs": b,
                       "latency (%)": 100 * (1 - ERES["HILO"]["latency_us"] / ERES[b]["latency_us"]),
                       "energy (%)": 100 * (1 - ERES["HILO"]["energy_uJ"] / ERES[b]["energy_uJ"]),
                       "area (%)": 100 * (1 - ERES["HILO"]["area_cells"] / ERES[b]["area_cells"])}
                      for b in TECHS5[:-1]]).style.format({c: "{:.2f}" for c in ("latency (%)", "energy (%)", "area (%)")}).hide(axis="index"))


## 6. 15 benchmarks
Dense mapping only needs the matrix size; the matrices are downloaded only to count all-zero columns (dropped when `DROP_ZERO_COLUMNS`). A matrix that cannot be downloaded is assumed to have no all-zero columns (marked *).

In [ ]:
import os, tarfile, urllib.request
# Download the 15 benchmark matrices (only to count all-zero columns). University of Florida mirror of
# SuiteSparse (sparse.tamu.edu returns 403), NIST Matrix Market where available.
BENCH15 = [("eris1176", 1176, 18552), ("cegb2919", 2919, 321543), ("raefsky1", 3242, 293409),
           ("fxm3_6", 5026, 94026), ("Na5", 5832, 305630), ("EX5", 6545, 295680),
           ("fp", 7548, 834222), ("ex40", 7740, 456188), ("benzene", 8219, 242669),
           ("bcsstk33", 8738, 591904), ("graham1", 9035, 335472), ("net25", 9520, 401200),
           ("bundle1", 10581, 770811), ("Si10H16", 17077, 875923), ("Goodwin_040", 17922, 561677)]
GROUPS = {"eris1176": "HB", "cegb2919": "HB", "raefsky1": "Simon", "fxm3_6": "Andrianov", "Na5": "PARSEC",
          "EX5": "JGD_SPG", "fp": "MKS", "ex40": "FIDAP", "benzene": "PARSEC", "bcsstk33": "HB",
          "graham1": "Graham", "net25": "Andrianov", "bundle1": "Lourakis", "Si10H16": "PARSEC",
          "Goodwin_040": "Goodwin"}
NIST = {"eris1176": "https://math.nist.gov/pub/MatrixMarket2/Harwell-Boeing/smtape/eris1176.mtx.gz",
        "bcsstk33": "https://math.nist.gov/pub/MatrixMarket2/Harwell-Boeing/bcsstruc5/bcsstk33.mtx.gz"}
os.makedirs("matrices", exist_ok=True)


def fetch(name):
    for path in (f"matrices/{name}.tar.gz", f"matrices/{name}.mtx.gz"):
        if os.path.exists(path):
            return "already here"
    tries = ([(NIST[name], f"matrices/{name}.mtx.gz")] if name in NIST else []) + \
            [(f"https://www.cise.ufl.edu/research/sparse/MM/{GROUPS[name]}/{name}.tar.gz", f"matrices/{name}.tar.gz")]
    errors = []
    for url, path in tries:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=120) as r, open(path, "wb") as f:
                f.write(r.read())
            if path.endswith(".mtx.gz") or tarfile.is_tarfile(path):
                return url
        except Exception as e:
            errors.append(f"{url.split('/')[2]}: {e}")
        if os.path.exists(path):
            os.remove(path)
    return "FAILED: " + " | ".join(errors)


for name, n, nnz in BENCH15:
    print(f"{name:12s} {fetch(name)}")


In [ ]:
import os, io, tarfile, math, numpy as np, pandas as pd, scipy.io, scipy.sparse as sp
import matplotlib as mpl, matplotlib.pyplot as plt
SHOW_AREA_PANEL = False            # True: add a normalized-area panel above latency and energy


def zero_columns(name, n):
    """All-zero columns of the real matrix (cached); None if the file is not here."""
    cache = f"matrices/{name}_zerocols.npy"
    if os.path.exists(cache):
        return int(np.load(cache))
    A = None
    if os.path.exists(f"matrices/{name}.mtx.gz"):
        A = scipy.io.mmread(f"matrices/{name}.mtx.gz")
    elif os.path.exists(f"matrices/{name}.tar.gz"):
        with tarfile.open(f"matrices/{name}.tar.gz") as tf:
            mtx = [m for m in tf.getmembers() if os.path.basename(m.name) == f"{name}.mtx"]
            A = scipy.io.mmread(io.BytesIO(tf.extractfile(mtx[0]).read()))
    if A is None:
        return None
    A = sp.csc_matrix(A); A.eliminate_zeros()
    z = int((np.diff(A.indptr) == 0).sum())
    np.save(cache, z)
    return z


# ---- 1. every technique on every benchmark ----
R15, rows, marks = {}, [], {}
for name, n, nnz in BENCH15:
    z = zero_columns(name, n)
    marks[name] = z is None
    kept = n - (z or 0) if DROP_ZERO_COLUMNS else n
    for t in TECHS5:
        r = run(t, n, kept)
        R15[(name, t)] = r
        rows.append({"matrix": name, "tech": t, "zero columns": "?" if z is None else z, "crossbars": r["crossbars"],
                     "tree levels": r["tree_levels"], "latency (us)": r["latency_us"], "energy (uJ)": r["energy_uJ"],
                     "area (M cells)": r["area_cells"] / 1e6, "area utilization (%)": 100 * r["area_utilization"]})
RAW15 = pd.DataFrame(rows)

# ---- 2. normalized to ULTRA, geometric mean ----
gmean = lambda v: float(np.exp(np.mean(np.log(v))))
names15 = [b[0] for b in BENCH15]
NORM = {(key, t): [R15[(nm, t)][metric] / R15[(nm, "ULTRA")][metric] for nm in names15]
        for key, metric in (("area", "area_cells"), ("lat", "latency_us"), ("en", "energy_uJ")) for t in TECHS5}
tab = {f"{t} norm {lab}": NORM[(key, t)] + [gmean(NORM[(key, t)])]
       for key, lab in (("lat", "latency"), ("en", "energy"), ("area", "area")) for t in TECHS5}
NORM_TABLE = pd.DataFrame(tab, index=names15 + ["GMean"])
print(f"Normalized to ULTRA (dense row mapping, shared x slot + working cells, tree reduction, {SET_MODE} SET, {LINKS} links)")
display(NORM_TABLE.style.format("{:.4f}"))

# ---- 3. overall improvement of HILO over the 15 benchmarks ----
rows = []
for key, lab, metric, unit, scale in (("lat", "Latency", "latency_us", "us", 1), ("en", "Energy", "energy_uJ", "uJ", 1),
                                      ("area", "Area", "area_cells", "M cells", 1e6)):
    g = {t: gmean(NORM[(key, t)]) for t in TECHS5}
    tot = {t: sum(R15[(nm, t)][metric] for nm in names15) / scale for t in TECHS5}
    for kind, d in (("geometric mean (norm. to ULTRA)", g), (f"total of 15 MVMs ({unit})", tot)):
        row = {"Metric": f"{lab}, {kind}", **d}
        for b in TECHS5[:-1]:
            row[f"HILO vs {b} (%)"] = 100 * (1 - d["HILO"] / d[b])
            row[f"HILO vs {b} (x)"] = d[b] / d["HILO"]
        rows.append(row)
IMP = pd.DataFrame(rows)
print("\nHILO improvement over the 15 benchmarks (positive % = HILO is lower)")
display(IMP.style.format({c: ("{:.4f}" if c.endswith("(x)") or c in TECHS5 else "{:,.2f}")
                          for c in IMP.columns if c != "Metric"}).hide(axis="index"))
